# IMPLEMENTATION OF xLSTM ARCHITECTURE ON XJTU DATASET

This notebook demonstrates the training and validation of the xLSTM-Transformer architecture across multiple window sizes on the XJTU-SY bearing dataset.

Author: GitHub Copilot
Date: 2026-04-18

In [ ]:
import os
import glob
import time
import gc
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Import Custom xLSTM Architecture
import sys
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from xLSTM_Implementation import xLSTM_Transformer_RUL

# Matplotlib Publication Standards
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'lines.linewidth': 2.0,
    'grid.alpha': 0.3
})
sns.set_style("whitegrid")


## 1. Global Configurations
Setup paths, experiment variables, and model hyperparameters.

In [ ]:
# Dataset Paths
DATASET_DIR = "../../XJTU-SY_Bearing_Datasets/Processed_Data/LSTM_Inputs"
RESULTS_DIR = "./xLSTM_Results"

os.makedirs(RESULTS_DIR, exist_ok=True)

# Experiment Setup
WINDOW_SIZES = [30, 40, 50, 70]
BEARING_LIFESPAN_TIME = 392_275 # Maximum lifespan for scaling RUL if needed
NUM_FEATURES = 15 # Should be dynamically parsed, but xLSTM defaults to 15

# Training Hyperparameters
EPOCHS = 100
BATCH_SIZE = 128
LEARNING_RATE = 0.001
EMBED_DIM = 32
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
DROPOUT = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Helper Functions
Metrics, plotting, and evaluation utilities.

In [ ]:
def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def relative_prediction_error(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def health_index_to_rul(hi, max_rul=BEARING_LIFESPAN_TIME):
    return np.clip(hi, 0, 1) * max_rul

def asymmetric_loss_scoring_function(y_pred, y_true, a=10, b=13):
    if not isinstance(y_pred, torch.Tensor):
        y_pred = torch.tensor(y_pred, dtype=torch.float32)
        y_true = torch.tensor(y_true, dtype=torch.float32)
    diff = y_pred - y_true
    loss = torch.where(
        diff < 0,
        torch.exp(-diff / a) - 1,
        torch.exp(diff / b) - 1
    )
    return float(loss.mean().item())

def plot_health_index_prediction(y_true, y_pred, title, save_path=None):
    plt.figure(figsize=(12, 6))
    plt.plot(y_true, label='True Health Index', color='black', alpha=0.8)
    plt.plot(y_pred, label='Predicted Health Index', color='gold', linestyle='--')
    plt.title(title, fontweight='bold')
    plt.xlabel('Time Steps')
    plt.ylabel('Health Index (0: Failed, 1: Normal)')
    plt.legend(loc='upper right')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


## 3. Training & Evaluation Pipeline
Looping over specified window sizes to train models and evaluate on validation bearings.

In [ ]:
# Summary Storage
summary_metrics = []
# For saving predictions across all experiments into an Excel file
excel_writer_path = os.path.join(RESULTS_DIR, "xLSTM_Predictions_Summary.xlsx")
excel_predictions = {} 

for ws in WINDOW_SIZES:
    print(f"\n{'='*60}\nStarting Experiment for Window Size: {ws}\n{'='*60}")
    
    # Create specific result directory for this window size
    ws_res_dir = os.path.join(RESULTS_DIR, f"window_size_{ws}")
    os.makedirs(ws_res_dir, exist_ok=True)
    
    # ----------------------------------------------------
    # A. LOAD TRAINING DATA
    # ----------------------------------------------------
    train_file = os.path.join(DATASET_DIR, f"processed_train_w{ws}.csv")
    if not os.path.exists(train_file):
        print(f"WARNING: Train file {train_file} not found. Skipping...")
        continue
        
    print(f"Loading training data: {train_file}")
    df_train = pd.read_csv(train_file)
    
    y_train = df_train['Target_RUL'].values
    
    # Exclude metadata columns to just get sequential features
    drop_cols = ['Target_RUL', 'Bearing_ID'] if 'Bearing_ID' in df_train.columns else ['Target_RUL']
    X_train_flat = df_train.drop(columns=drop_cols).values
    
    # Reshape (Samples, Window_Size, Features)
    num_samples = X_train_flat.shape[0]
    num_features = X_train_flat.shape[1] // ws
    X_train_3d = X_train_flat.reshape(num_samples, ws, num_features)
    
    X_train_tensor = torch.tensor(X_train_3d, dtype=torch.float32).to(DEVICE)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(DEVICE)
    
    # Free memory
    del df_train, X_train_flat
    gc.collect()
    
    # ----------------------------------------------------
    # B. INITIALIZE MODEL & TRAIN (xLSTM)
    # ----------------------------------------------------
    model = xLSTM_Transformer_RUL(
        num_features=num_features,
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        num_decoder_layers=NUM_DECODER_LAYERS,
        dropout=DROPOUT
    ).to(DEVICE)
    
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()
    
    print("Training Model...")
    model.train()
    
    # Simplified Dataloader 
    dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    best_loss = float('inf')
    hi_loss_history = []
    
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        for batch_x, batch_y in dataloader:
            optimizer.zero_grad()
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(dataloader)
        hi_loss_history.append(avg_loss)
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            # Save best state locally in RAM (or disk)
            best_model_state = copy.deepcopy(model.state_dict())
            
        if (epoch + 1) % 20 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] | MSE Loss: {avg_loss:.6f}")
    
    # Load best weights
    model.load_state_dict(best_model_state)
    torch.save(model.state_dict(), os.path.join(ws_res_dir, "best_xlstm_model.pth"))
    
    # Plot Training Loss
    plt.figure(figsize=(10, 5))
    plt.plot(hi_loss_history, label='Train MSE Loss', color='red')
    plt.title(f'Training Loss History (Window Size = {ws})')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(ws_res_dir, "training_loss_history.png"), dpi=300)
    plt.close()
    
    # Free Training GPU Tensor Memory
    del X_train_tensor, y_train_tensor, dataset, dataloader
    torch.cuda.empty_cache()
    gc.collect()
    
    # ----------------------------------------------------
    # C. EVALUATION ON VALIDATION BEARINGS
    # ----------------------------------------------------
    val_files = glob.glob(os.path.join(DATASET_DIR, f"processed_val_*_w{ws}.csv"))
    
    ws_predictions_df = pd.DataFrame()
    ws_metrics = []
    
    model.eval()
    with torch.no_grad():
        for v_file in val_files:
            b_id = re.search(r"processed_val_(.*)_w\d+\.csv", os.path.basename(v_file)).group(1)
            print(f"Evaluating on {b_id} (Window={ws})...")
            
            df_val = pd.read_csv(v_file)
            y_val_true = df_val['Target_RUL'].values
            val_drop_cols = ['Target_RUL', 'Bearing_ID'] if 'Bearing_ID' in df_val.columns else ['Target_RUL']
            X_val_flat = df_val.drop(columns=val_drop_cols).values
            
            # Reshape
            ns = X_val_flat.shape[0]
            nf = X_val_flat.shape[1] // ws
            X_val_3d = X_val_flat.reshape(ns, ws, nf)
            
            X_val_tensor = torch.tensor(X_val_3d, dtype=torch.float32).to(DEVICE)
            
            # Predict
            y_val_pred = model(X_val_tensor).cpu().numpy().flatten()
            
            # Metrics
            rmse_val = rmse_score(y_val_true, y_val_pred)
            mae_val = mean_absolute_error(y_val_true, y_val_pred)
            r2_val = r2_score(y_val_true, y_val_pred)
            rpe_val = relative_prediction_error(y_val_true, y_val_pred)
            asym_loss = asymmetric_loss_scoring_function(y_val_pred, y_val_true)
            
            ws_metrics.append({
                'Bearing_ID': b_id,
                'Window_Size': ws,
                'RMSE': rmse_val,
                'MAE': mae_val,
                'R2': r2_val,
                'RPE(%)': rpe_val,
                'Asymmetric_Loss': asym_loss
            })
            
            # Sub-dataframe for this bearing
            b_pred_df = pd.DataFrame({
                'Bearing_ID': b_id,
                'Time_Step': np.arange(len(y_val_true)),
                'True_Health_Index': y_val_true,
                'Predicted_Health_Index': y_val_pred,
                'True_RUL': health_index_to_rul(y_val_true),
                'Predicted_RUL': health_index_to_rul(y_val_pred)
            })
            ws_predictions_df = pd.concat([ws_predictions_df, b_pred_df], ignore_index=True)
            
            # Plot
            plot_title = f"{b_id} Health Index Prediction (xLSTM - WS={ws})"
            plot_save_path = os.path.join(ws_res_dir, f"prediction_plot_{b_id}.png")
            plot_health_index_prediction(y_val_true, y_val_pred, plot_title, plot_save_path)
            
            del df_val, X_val_flat, X_val_3d, X_val_tensor
            gc.collect()
            
    # Save WS Predictions
    ws_predictions_df.to_csv(os.path.join(ws_res_dir, f"predictions_ws_{ws}.csv"), index=False)
    excel_predictions[f"WS_{ws}"] = ws_predictions_df
    
    # Store average metrics for this Window Size across all bearings
    if len(ws_metrics) > 0:
        ws_metrics_df = pd.DataFrame(ws_metrics)
        ws_metrics_df.to_csv(os.path.join(ws_res_dir, f"metrics_ws_{ws}.csv"), index=False)
        mean_metrics = ws_metrics_df.mean(numeric_only=True)
        
        summary_metrics.append({
            'Window_Size': ws,
            'Mean_RMSE': mean_metrics['RMSE'],
            'Mean_MAE': mean_metrics['MAE'],
            'Mean_R2': mean_metrics['R2'],
            'Mean_RPE(%)': mean_metrics['RPE(%)'],
            'Mean_Asymmetric_Loss': mean_metrics['Asymmetric_Loss']
        })

# Export Combined Predictions to Excel
print(f"\nSaving all predictions to summary Excel: {excel_writer_path}")
with pd.ExcelWriter(excel_writer_path) as writer:
    for sheet_name, df in excel_predictions.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        
print("All Experiments Completed Succesfully!")


## 4. Final Window Size Optimization Analysis
Comparing evaluating mean MSE/RMSE across the 4 window size variants to deduce the optimal length.

In [ ]:
summary_df = pd.DataFrame(summary_metrics)

if not summary_df.empty:
    display(summary_df)
    summary_df.to_csv(os.path.join(RESULTS_DIR, "xLSTM_WindowSize_Comparison.csv"), index=False)
    
    plt.figure(figsize=(10, 6))
    plt.plot(summary_df['Window_Size'], summary_df['Mean_RMSE'], marker='o', linestyle='-', color='purple', linewidth=2.5, markersize=8)
    
    # Annotate points
    for i, row in summary_df.iterrows():
        plt.annotate(f"{row['Mean_RMSE']:.4f}", 
                     (row['Window_Size'], row['Mean_RMSE']),
                     textcoords="offset points", 
                     xytext=(0,10), 
                     ha='center')
                     
    plt.title('Mean RMSE vs Window Size', fontweight='bold', pad=15)
    plt.xlabel('Window Size')
    plt.ylabel('Mean Validation RMSE')
    plt.xticks(summary_df['Window_Size'].values)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "optim_rmse_vs_windowsize.png"), dpi=300)
    plt.show()
    
    optimal_ws = summary_df.loc[summary_df['Mean_RMSE'].idxmin()]['Window_Size']
    print(f"\n>>> Based on Mean Validation RMSE, the Optimal Window Size is: {int(optimal_ws)} <<<")
else:
    print("No summary metrics available to analyze.")
